# Consumer Retail Analysis

## Python Data Preparation & Dashboard Analysis

This notebook prepares the customer shopping dataset for the Consumer Retail Analysis project.

### Dashboard-aligned analysis
- Total Customers
- Average Purchase Amount
- Average Review Rating
- Subscription Status
- Revenue by Season
- Revenue by Category
- Sales by Gender
- Payment Method Split
- Shipping Type Preference

> **Dataset requirement:** Place `customer_shopping_behavior.csv` in the same folder as this notebook before running the notebook.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Load the Dataset

In [ ]:
# Keep the dataset path relative so the notebook works on another computer.
DATA_FILE = "customer_shopping_behavior.csv"

df = pd.read_csv(DATA_FILE)

print(f"Dataset loaded successfully: {df.shape[0]:,} rows and {df.shape[1]} columns")
df.head()


In [ ]:
# Basic structure
df.info()


In [ ]:
# Numerical summary
df.describe()


In [ ]:
# Missing-value check
missing_values = df.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]


## 2. Clean and Standardize the Data

The original project notebook used snake_case column names, handled missing Review Rating values, 
created customer age groups, converted purchase frequency into days, and checked the relationship 
between Discount Applied and Promo Code Used. The steps below keep that project logic while removing 
the broken/error cells from the original notebook.


In [ ]:
# Standardize column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(r"[^a-z0-9]+", "_", regex=True)
      .str.strip("_")
)

df.columns.tolist()


In [ ]:
# Ensure key numeric columns have numeric data types
numeric_columns = [
    "customer_id",
    "age",
    "purchase_amount_usd",
    "review_rating",
    "previous_purchases",
]

for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

df[["customer_id", "age", "purchase_amount_usd", "review_rating", "previous_purchases"]].head()


In [ ]:
# Handle missing Review Rating values using the median rating of each product category.
# This follows the methodology described in the original project report.
if df["review_rating"].isna().any():
    category_medians = df.groupby("category")["review_rating"].transform("median")
    df["review_rating"] = df["review_rating"].fillna(category_medians)

print("Remaining missing Review Rating values:", df["review_rating"].isna().sum())


In [ ]:
# Create age groups using quartiles, matching the original notebook's approach.
age_labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
df["age_group"] = pd.qcut(
    df["age"],
    q=4,
    labels=age_labels,
    duplicates="drop"
)

df[["age", "age_group"]].head(10)


In [ ]:
# Convert purchase frequency to an approximate number of days.
frequency_mapping = {
    "Weekly": 7,
    "Fortnightly": 14,
    "Bi-Weekly": 14,
    "Monthly": 30,
    "Every 3 Months": 90,
    "Quarterly": 90,
    "Annually": 365,
}

df["purchase_frequency_days"] = df["frequency_of_purchases"].map(frequency_mapping)

df[["frequency_of_purchases", "purchase_frequency_days"]].drop_duplicates().sort_values(
    "purchase_frequency_days"
)


In [ ]:
# Check whether Discount Applied and Promo Code Used contain the same information.
comparison = (
    df[["discount_applied", "promo_code_used"]]
    .value_counts(dropna=False)
    .reset_index(name="count")
)

comparison


In [ ]:
# The source project treated Promo Code Used as redundant and dropped it.
# We only drop it if it exists.
if "promo_code_used" in df.columns:
    df = df.drop(columns=["promo_code_used"])

print("Final columns:")
print(df.columns.tolist())


## 3. Dashboard KPI Calculations

In [ ]:
# KPI values used by the dashboard
total_customers = len(df)
average_purchase = df["purchase_amount_usd"].mean()
average_rating = df["review_rating"].mean()

print(f"Total Customers: {total_customers:,}")
print(f"Average Purchase Amount: ${average_purchase:,.2f}")
print(f"Average Review Rating: {average_rating:.2f}")


## 4. Subscription Status

In [ ]:
subscription_summary = (
    df["subscription_status"]
    .value_counts()
    .rename_axis("subscription_status")
    .reset_index(name="customers")
)

subscription_summary["percentage"] = (
    subscription_summary["customers"] / total_customers * 100
).round(2)

subscription_summary


## 5. Revenue by Season

In [ ]:
revenue_by_season = (
    df.groupby("season", as_index=False)["purchase_amount_usd"]
      .sum()
      .sort_values("purchase_amount_usd", ascending=False)
)

revenue_by_season


In [ ]:
# Dashboard-style seasonal revenue chart
season_order = ["Fall", "Spring", "Winter", "Summer"]
season_plot = revenue_by_season.copy()
season_plot["season"] = pd.Categorical(
    season_plot["season"], categories=season_order, ordered=True
)
season_plot = season_plot.sort_values("season")

plt.figure(figsize=(8, 4))
plt.bar(season_plot["season"], season_plot["purchase_amount_usd"])
plt.title("Revenue by Season")
plt.xlabel("Season")
plt.ylabel("Revenue (USD)")
plt.tight_layout()
plt.show()


## 6. Revenue by Category

In [ ]:
revenue_by_category = (
    df.groupby("category", as_index=False)["purchase_amount_usd"]
      .sum()
      .sort_values("purchase_amount_usd", ascending=False)
)

revenue_by_category


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(revenue_by_category["category"], revenue_by_category["purchase_amount_usd"])
plt.title("Revenue by Category")
plt.xlabel("Category")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 7. Sales by Gender

In [ ]:
sales_by_gender = (
    df.groupby("gender", as_index=False)["purchase_amount_usd"]
      .sum()
      .sort_values("purchase_amount_usd", ascending=False)
)

sales_by_gender


In [ ]:
plt.figure(figsize=(7, 4))
plt.barh(sales_by_gender["gender"], sales_by_gender["purchase_amount_usd"])
plt.title("Sales by Gender")
plt.xlabel("Sales (USD)")
plt.ylabel("Gender")
plt.tight_layout()
plt.show()


## 8. Payment Method Split

In [ ]:
payment_summary = (
    df["payment_method"]
    .value_counts()
    .rename_axis("payment_method")
    .reset_index(name="transactions")
)

payment_summary["percentage"] = (
    payment_summary["transactions"] / total_customers * 100
).round(2)

payment_summary


In [ ]:
plt.figure(figsize=(7, 7))
plt.pie(
    payment_summary["transactions"],
    labels=payment_summary["payment_method"],
    autopct="%1.1f%%",
    startangle=90
)
plt.title("Payment Method Split")
plt.tight_layout()
plt.show()


## 9. Shipping Type Preference

In [ ]:
shipping_summary = (
    df["shipping_type"]
    .value_counts()
    .rename_axis("shipping_type")
    .reset_index(name="transactions")
)

shipping_summary["percentage"] = (
    shipping_summary["transactions"] / total_customers * 100
).round(2)

shipping_summary


In [ ]:
plt.figure(figsize=(9, 4))
plt.bar(shipping_summary["shipping_type"], shipping_summary["transactions"])
plt.title("Shipping Type Preference")
plt.xlabel("Shipping Type")
plt.ylabel("Transactions")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


## 10. Supporting Analysis

In [ ]:
# Top-rated products from the cleaned dataset
top_rated_products = (
    df.groupby("item_purchased", as_index=False)["review_rating"]
      .mean()
      .sort_values("review_rating", ascending=False)
      .head(5)
)

top_rated_products


In [ ]:
# Revenue by gender — detailed values used to support the dashboard
gender_revenue = (
    df.groupby("gender", as_index=False)["purchase_amount_usd"]
      .sum()
      .sort_values("purchase_amount_usd", ascending=False)
)

gender_revenue


In [ ]:
# Products with the highest discount dependency
discount_dependency = (
    df.assign(
        discount_flag=df["discount_applied"].astype(str).str.lower().eq("yes")
    )
    .groupby("item_purchased")["discount_flag"]
    .mean()
    .mul(100)
    .reset_index(name="discount_rate")
    .sort_values("discount_rate", ascending=False)
    .head(5)
)

discount_dependency


In [ ]:
# Customer segmentation based on previous purchase history.
# Thresholds follow the original project report:
# New = 1 purchase, Returning = 2–10, Loyal = >10.
def customer_segment(purchases):
    if purchases == 1:
        return "New"
    elif purchases <= 10:
        return "Returning"
    return "Loyal"

df["customer_segment"] = df["previous_purchases"].apply(customer_segment)

customer_segments = (
    df["customer_segment"]
    .value_counts()
    .rename_axis("customer_segment")
    .reset_index(name="customers")
)

customer_segments


## 11. Export Cleaned Data

In [ ]:
# Save the cleaned dataset for SQL / Power BI work.
OUTPUT_FILE = "customer_shopping_behavior_cleaned.csv"

df.to_csv(OUTPUT_FILE, index=False)

print(f"Cleaned dataset saved as: {OUTPUT_FILE}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")


## 12. Optional PostgreSQL Upload

The original notebook contained hard-coded PostgreSQL credentials and repeated connection cells.
Those cells were intentionally replaced with environment variables so credentials are not exposed
in a GitHub repository.

Set these variables before running the optional upload section:

- `PGUSER`
- `PGPASSWORD`
- `PGHOST`
- `PGPORT`
- `PGDATABASE`

Install the PostgreSQL driver if required:

```bash
pip install sqlalchemy psycopg2-binary
```


In [ ]:
# Optional PostgreSQL upload
# This section is disabled by default. Uncomment/run it only after setting your environment variables.

import os

PGUSER = os.getenv("PGUSER")
PGPASSWORD = os.getenv("PGPASSWORD")
PGHOST = os.getenv("PGHOST", "localhost")
PGPORT = os.getenv("PGPORT", "5432")
PGDATABASE = os.getenv("PGDATABASE")

if all([PGUSER, PGPASSWORD, PGDATABASE]):
    try:
        from sqlalchemy import create_engine

        engine = create_engine(
            f"postgresql+psycopg2://{PGUSER}:{PGPASSWORD}@{PGHOST}:{PGPORT}/{PGDATABASE}"
        )

        table_name = "customer"
        df.to_sql(table_name, engine, if_exists="replace", index=False)

        print(f"Data successfully loaded into PostgreSQL table: {table_name}")
    except Exception as e:
        print("PostgreSQL upload was not completed.")
        print("Check your driver, database settings, and connection.")
        print("Error:", e)
else:
    print("PostgreSQL upload skipped: database environment variables are not configured.")


## Final Dashboard Alignment

The cleaned notebook is designed to support the supplied **Customer Analytics Dashboard**.

### Main dashboard measures
- **3.9K** customers
- **$59.76** average purchase amount
- **3.75** average review rating

### Main dashboard visuals
- Subscription Status
- Revenue by Season
- Revenue by Category
- Sales by Gender
- Payment Method Split
- Shipping Type Preference

The notebook calculates these measures from the dataset rather than hard-coding the dashboard values.
